## Train SimPLe on the Freeway enviorment

Stored checkpoings for Freeway in https://drive.google.com/drive/folders/12LOR9OaQqX6ng7x63pNHH18yt-yn06Y7?usp=drive_link

To use, create a shortcut to this folder on your Google Drive, then mount.

In [ ]:
# get saved checkpoints from google drive

from google.colab import drive
drive.mount('/content/drive')

In [3]:
# clone repo

# !git clone https://github.com/wcohen2027/cs5180-dqn-atari-games-final-project.git

!cd cs5180-dqn-atari-games-final-project/SimPLe && git pull

Cloning into 'cs5180-dqn-atari-games-final-project'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 77 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 4.63 MiB | 11.90 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Already up to date.


In [ ]:
# Install dependencies
!pip install -q "gymnasium[atari]" ale-py tqdm wandb
!pip install -q "autorom[accept-rom-license]"
!pip install -q cloudpickle

In [ ]:
# Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# run SimPLe training as a subprocess, capture per-epoch rewards
import subprocess
import sys
import re
import json
import matplotlib.pyplot as plt
import numpy as np
import torch

SIMPLE_DIR = '/content/SimPLe'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

def run_simple(env_name, epochs=15, agents=16, device=None,
               extra_args=None, output_log=None):
    """
    Launch SimPLe training for `env_name` and return a list of
    mean eval rewards (one per epoch).

    __main__.py prints a SIMPLE_EVAL line after each epoch evaluation;
    we parse those here.
    """
    if device is None:
        device = DEVICE

    cmd = [
        sys.executable, '-m', 'simple',
        '--env-name', env_name,
        '--epochs', str(epochs),
        '--agents', str(agents),
        '--device', device,
        '--save-models',
    ]
    if extra_args:
        cmd.extend([str(a) for a in extra_args])

    env = os.environ.copy()
    env['PYTHONPATH'] = ':'.join([
        SIMPLE_DIR,
        os.path.join(SIMPLE_DIR, 'atari_utils'),
        os.path.join(SIMPLE_DIR, 'a2c_ppo_acktr'),
        env.get('PYTHONPATH', ''),
    ])

    rewards = []
    log_lines = []

    print(f'\n=== Training SimPLe on {env_name} ({epochs} epochs, device={device}) ===')
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=SIMPLE_DIR,
        env=env,
    )

    for line in process.stdout:
        print(line, end='')
        log_lines.append(line)
        m = re.search(r'SIMPLE_EVAL eval_score_mean=(-?[\d.]+)', line)
        if m:
            rewards.append(float(m.group(1)))

    process.wait()

    if output_log:
        with open(output_log, 'w') as f:
            f.writelines(log_lines)

    if process.returncode != 0:
        print(f'WARNING: process exited with code {process.returncode}')

    return rewards

In [ ]:
import os

# SimPLe's filepath
SIMPLE_DIR = '/content/cs5180-dqn-atari-games-final-project/SimPLe'

# Create a directory on Drive for this run
drive_models = '/content/drive/MyDrive/SimPLe/models_pong'
os.makedirs(drive_models, exist_ok=True)

# Symlink so SimPLe writes directly to Google Drive
models_local = os.path.join(SIMPLE_DIR, 'models')
if os.path.exists(models_local) and not os.path.islink(models_local):
    import shutil
    shutil.copytree(models_local, drive_models, dirs_exist_ok=True)
    shutil.rmtree(models_local)

if not os.path.exists(models_local):
    os.symlink(drive_models, models_local)

print(f'Models will be saved to: {drive_models}')

In [ ]:
# Freeway epoch 1 -> 15
freeway_rewards = run_simple(
    env_name='Freeway',
    epochs=15,
    output_log='/content/freeway_log.txt',
    extra_args=['--ppo-gamma', 0.95, '--ppo-eval-period', 1000,
                 '--save-models', '--start-epoch', 0],
)
print('Freeway rewards per epoch:', freeway_rewards)

In [ ]:
# graph returns of Freeway

fig, ax = plt.subplots(figsize=(8, 4))
iterations = list(range(1, len(freeway_rewards) + 1))
ax.plot(iterations, freeway_rewards, label='SimPLe')
ax.set_title('Freeway')
ax.set_xlabel('iteration')
ax.set_ylabel('train_episode_returns')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('/content/simple_results.png', dpi=150)
plt.show()
print('Plot saved to /content/simple_results.png')